<a href="https://colab.research.google.com/github/rahiakela/deep-learning-research-and-practice/blob/main/neural-networks-from-scratch-in-python/05_introducing_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

Our next step is to determine how to adjust the weights and biases to decrease the loss.

Finding an intelligent way to adjust the neurons's input's weights and biases to minimize loss
is the main difficulty of neural networks.

In [ ]:
!pip install nnfs

In [2]:
import numpy as np

import nnfs
from nnfs.datasets import spiral_data
from nnfs.datasets import vertical_data

import math
import matplotlib.pyplot as plt

## Dense Layer

In [3]:
class DenseLayer:

  # Initialize weights and biases
  def __init__(self, n_inputs, n_neurons) -> None:
    # Note that we’re initializing weights to be (inputs, neurons), rather than ( neurons, inputs)
    self.weights = 0.01 * np.random.randn(n_inputs, n_neurons) # Gaussian distribution with a mean of 0 and a variance of 1
    self.bias = np.zeros((1, n_neurons))

  # Forward pass: Here, we pass data through a model from beginning to end
  def forward(self, inputs):
    # Calculate output values from inputs, weights and biases
    self.output = np.dot(inputs, self.weights) + self.bias

## ReLU Activation

In [4]:
class ReLU():
  def forward(self, inputs):
    # Calculate output values from input
    self.output = np.maximum(0, inputs)

## Softmax Activation

In [5]:
class Softmax:
  def forward(self, inputs):
    # Get unnormalized probabilities
    exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))

    # Normalize them for each sample
    probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
    self.output = probabilities

## Categorical Cross-Entropy Loss

In [6]:
class Loss:
  # Calculates the data and regularization losses
  # given model output and ground truth values
  def calculate(self, output, y):
    # Calculate sample losses
    sample_losses = self.forward(output, y)
    # Calculate mean loss
    data_loss = np.mean(sample_losses)
    # Return loss
    return data_loss

In [7]:
# Cross-entropy loss
class CategoricalCrossentropyLoss(Loss):
  def forward(self, y_pred, y_true):
    # Number of samples in a batch
    samples = len(y_pred)
    # Clip data to prevent division by 0
    # Clip both sides to not drag mean towards any value
    y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

    # Probabilities for target values - only if categorical labels
    if len(y_true.shape) == 1:
      correct_confidences = y_pred_clipped[range(samples), y_true]
    elif len(y_true.shape) == 2: # Mask values - only for one-hot encoded labels
      correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)

    # Losses
    negative_log_likelihoods = -np.log(correct_confidences)
    return negative_log_likelihoods

## Neural Network for Vertical data

In [8]:
# Create dataset
X, y = vertical_data(samples=100, classes=3)

# Create Dense layer with 2 input features and 3 output values
dense_layer1 = DenseLayer(2, 3)

# Create ReLU activation (to be used with Dense layer):
relu_activation = ReLU()

# Create second Dense layer with 3 input features (as we take output
# of previous layer here) and 3 output values
dense_layer2 = DenseLayer(3, 3)

# Create Softmax activation (to be used with Dense layer):
softmax_activation = Softmax()

# Create loss function
loss_function = CategoricalCrossentropyLoss()

# variables to track the best loss and the associated weights and biases
lowest_loss = 9999999 # some initial value
best_dense1_weights = dense_layer1.weights.copy()
best_dense1_biases = dense_layer1.bias.copy()
best_dense2_weights = dense_layer2.weights.copy()
best_dense2_biases = dense_layer2.bias.copy()

# Now we iterate as many times as desired, pick random values for weights and biases
for iteration in range(10000):
  # Generate a new set of weights for iteration
  dense_layer1.weights = 0.05 * np.random.randn(2, 3)
  dense_layer1.bias = 0.05 * np.random.randn(1, 3)
  dense_layer2.weights = 0.05 * np.random.randn(3, 3)
  dense_layer2.bias = 0.05 * np.random.randn(1, 3)

  # Perform a forward pass of the training data through this layer
  dense_layer1.forward(X)
  relu_activation.forward(dense_layer1.output)
  dense_layer2.forward(relu_activation.output)
  softmax_activation.forward(dense_layer2.output)

  # Perform a forward pass through loss function
  # it takes the output of second dense layer here and returns loss
  loss = loss_function.calculate(softmax_activation.output, y)

  # Calculate accuracy from output of softmax_activation and targets
  predictions = np.argmax(softmax_activation.output, axis=1) # calculate values along first axis
  accuracy = np.mean(predictions==y)

  # If loss is smaller - print and save weights and biases aside
  if loss < lowest_loss:
    print(f'New set of weights found, iteration: {iteration}, loss: {loss}, accuracy: {accuracy}')
    best_dense1_weights = dense_layer1.weights.copy()
    best_dense1_biases = dense_layer1.bias.copy()
    best_dense2_weights = dense_layer2.weights.copy()
    best_dense2_biases = dense_layer2.bias.copy()
    lowest_loss = loss

New set of weights found, iteration: 0, loss: 1.100351407250404, accuracy: 0.3333333333333333
New set of weights found, iteration: 2, loss: 1.098496604042293, accuracy: 0.3333333333333333
New set of weights found, iteration: 21, loss: 1.0982595149279044, accuracy: 0.3333333333333333
New set of weights found, iteration: 24, loss: 1.0980806582873526, accuracy: 0.66
New set of weights found, iteration: 117, loss: 1.0973686461259335, accuracy: 0.6033333333333334
New set of weights found, iteration: 159, loss: 1.0967636000152041, accuracy: 0.6033333333333334
New set of weights found, iteration: 1679, loss: 1.0963457224698958, accuracy: 0.3333333333333333
New set of weights found, iteration: 7546, loss: 1.09599560741767, accuracy: 0.3333333333333333


Instead of setting parameters with randomly-chosen values each iteration, apply a fraction of these values to parameters.

With this, weights will be updated from what currently yields us the lowest loss instead of aimlessly randomly.

If the adjustment decreases loss, we will make it the new point to adjust from. If loss instead increases due to the adjustment, then we will revert to the previous point.

In [9]:
# Create dataset
X, y = vertical_data(samples=100, classes=3)

# Create Dense layer with 2 input features and 3 output values
dense_layer1 = DenseLayer(2, 3)

# Create ReLU activation (to be used with Dense layer):
relu_activation = ReLU()

# Create second Dense layer with 3 input features (as we take output
# of previous layer here) and 3 output values
dense_layer2 = DenseLayer(3, 3)

# Create Softmax activation (to be used with Dense layer):
softmax_activation = Softmax()

# Create loss function
loss_function = CategoricalCrossentropyLoss()

# variables to track the best loss and the associated weights and biases
lowest_loss = 9999999 # some initial value
best_dense1_weights = dense_layer1.weights.copy()
best_dense1_biases = dense_layer1.bias.copy()
best_dense2_weights = dense_layer2.weights.copy()
best_dense2_biases = dense_layer2.bias.copy()

# Now we iterate as many times as desired, pick random values for weights and biases
for iteration in range(10000):
  # Update weights with some small random values
  dense_layer1.weights += 0.05 * np.random.randn(2, 3)
  dense_layer1.bias += 0.05 * np.random.randn(1, 3)
  dense_layer2.weights += 0.05 * np.random.randn(3, 3)
  dense_layer2.bias += 0.05 * np.random.randn(1, 3)

  # Perform a forward pass of the training data through this layer
  dense_layer1.forward(X)
  relu_activation.forward(dense_layer1.output)
  dense_layer2.forward(relu_activation.output)
  softmax_activation.forward(dense_layer2.output)

  # Perform a forward pass through loss function
  # it takes the output of second dense layer here and returns loss
  loss = loss_function.calculate(softmax_activation.output, y)

  # Calculate accuracy from output of softmax_activation and targets
  predictions = np.argmax(softmax_activation.output, axis=1) # calculate values along first axis
  accuracy = np.mean(predictions==y)

  # If loss is smaller - print and save weights and biases aside
  if loss < lowest_loss:
    print(f'New set of weights found, iteration: {iteration}, loss: {loss}, accuracy: {accuracy}')
    best_dense1_weights = dense_layer1.weights.copy()
    best_dense1_biases = dense_layer1.bias.copy()
    best_dense2_weights = dense_layer2.weights.copy()
    best_dense2_biases = dense_layer2.bias.copy()
    lowest_loss = loss
  else:
    # Revert weights and biases
    best_dense1_weights = dense_layer1.weights.copy()
    best_dense1_biases = dense_layer1.bias.copy()
    best_dense2_weights = dense_layer2.weights.copy()
    best_dense2_biases = dense_layer2.bias.copy()

New set of weights found, iteration: 0, loss: 1.0979698018461885, accuracy: 0.3333333333333333
New set of weights found, iteration: 11, loss: 1.0962780036492532, accuracy: 0.3333333333333333
New set of weights found, iteration: 14, loss: 1.0911751032756514, accuracy: 0.6466666666666666


## Neural Network for Spiral data

In [10]:
# Create dataset
X, y = spiral_data(samples=100, classes=3)

# Create Dense layer with 2 input features and 3 output values
dense_layer1 = DenseLayer(2, 3)

# Create ReLU activation (to be used with Dense layer):
relu_activation = ReLU()

# Create second Dense layer with 3 input features (as we take output
# of previous layer here) and 3 output values
dense_layer2 = DenseLayer(3, 3)

# Create Softmax activation (to be used with Dense layer):
softmax_activation = Softmax()

# Create loss function
loss_function = CategoricalCrossentropyLoss()

# variables to track the best loss and the associated weights and biases
lowest_loss = 9999999 # some initial value
best_dense1_weights = dense_layer1.weights.copy()
best_dense1_biases = dense_layer1.bias.copy()
best_dense2_weights = dense_layer2.weights.copy()
best_dense2_biases = dense_layer2.bias.copy()

# Now we iterate as many times as desired, pick random values for weights and biases
for iteration in range(10000):
  # Update weights with some small random values
  dense_layer1.weights += 0.05 * np.random.randn(2, 3)
  dense_layer1.bias += 0.05 * np.random.randn(1, 3)
  dense_layer2.weights += 0.05 * np.random.randn(3, 3)
  dense_layer2.bias += 0.05 * np.random.randn(1, 3)

  # Perform a forward pass of the training data through this layer
  dense_layer1.forward(X)
  relu_activation.forward(dense_layer1.output)
  dense_layer2.forward(relu_activation.output)
  softmax_activation.forward(dense_layer2.output)

  # Perform a forward pass through loss function
  # it takes the output of second dense layer here and returns loss
  loss = loss_function.calculate(softmax_activation.output, y)

  # Calculate accuracy from output of softmax_activation and targets
  predictions = np.argmax(softmax_activation.output, axis=1) # calculate values along first axis
  accuracy = np.mean(predictions==y)

  # If loss is smaller - print and save weights and biases aside
  if loss < lowest_loss:
    print(f'New set of weights found, iteration: {iteration}, loss: {loss}, accuracy: {accuracy}')
    best_dense1_weights = dense_layer1.weights.copy()
    best_dense1_biases = dense_layer1.bias.copy()
    best_dense2_weights = dense_layer2.weights.copy()
    best_dense2_biases = dense_layer2.bias.copy()
    lowest_loss = loss
  else:
    # Revert weights and biases
    best_dense1_weights = dense_layer1.weights.copy()
    best_dense1_biases = dense_layer1.bias.copy()
    best_dense2_weights = dense_layer2.weights.copy()
    best_dense2_biases = dense_layer2.bias.copy()

New set of weights found, iteration: 0, loss: 1.0987762866794983, accuracy: 0.3333333333333333
New set of weights found, iteration: 1, loss: 1.0984695893746068, accuracy: 0.3333333333333333
New set of weights found, iteration: 71, loss: 1.0984430159278098, accuracy: 0.35333333333333333
New set of weights found, iteration: 74, loss: 1.0972409672116838, accuracy: 0.36333333333333334
New set of weights found, iteration: 75, loss: 1.0938198328149875, accuracy: 0.3466666666666667
New set of weights found, iteration: 76, loss: 1.0927268978546794, accuracy: 0.3566666666666667
New set of weights found, iteration: 101, loss: 1.0926970007313477, accuracy: 0.36333333333333334
New set of weights found, iteration: 102, loss: 1.0916592050247709, accuracy: 0.37


This training session ended with almost no progress. Loss decreased slightly and accuracy is barely above the initial value.

It turns out hard problems are hard for a reason, and we need to approach this problem more intelligently.